# 02 · Prepare the week from `data_raw/` (notebooks3: new export names, dates t1 / t2, the latest campaign, run size)

**What notebooks3 changes here** (the rest is the notebooks2 notebook):
- **The new export names:** `subs_btl_test` → `subs_btl_auto`, `dv_segment` → `DEVICEBEHAVIORSEGMENTATION.dv_segment`,
  `SEGMENT_Conflicting_Offers_h` → `conflicting_offers_h`, and the 11 model files without `_TD`. The exports that are
  not used are ignored: `target_new_april26`, `tenure`, `DAIM_PROPENSITY_REACTIVATION_30D_404_SCORING_filterd_nationality`.
- **The dates:** **D = t1** = the most recent day of the data (the last purchase day of `subs_btl`), the anchor of
  the `*_score` tables; **T = t2 = t1 − 7**, the anchor of the `*_auto` tables, where the model trains. Last week's
  purchases are counted from t2 to t1.
- **`targeting_dataiku_auto` holds the whole history:** step 4c keeps, per line, its most recent row, and only if it
  belongs to the latest campaign (`LAST_CAMPAIGN_DAYS`). Without this, a customer targeted weeks ago would count as
  "targeted last week".
- **`RUN_SIZE`:** `1_000`, `5_000` or `10_000` = also a test copy of that many target lines; `"all"` = the full week.
- **`FIX_HEADERS = True`:** a file without a header gets the contract header (when its columns and IDs match).
- **The time of every step**, its estimate and the time left are printed as `[time]` lines.

You copy **every exported table as it is** into one flat folder, `data/data_raw/` (names as in Dataiku, with or
without the `_hdfs` suffix, `.csv` or `.csv.gz`). This notebook:

1. **Reads `data_raw/`**: each file gets its contract name (`base_prepaid_hdfs` → `base_prepaid`,
   `activities_w1_score_hfds` → `activities_w1_score`, typos and case fixed), header checked, nothing invented.
2. **Lists the tables**: the 41 of the NBA logic (training `*_auto`, scoring `*_score`, models, offers), what is
   missing, what is optional.
3. **Checks the two dates**: the run date **D** (scoring, `*_score`) read from the data itself, and the anchor
   **T** (training, `*_auto`) = the last Thursday on or before D; every window of both; whether the dated tables
   (`targeting_dataiku_auto`, `subs_btl_auto`, `targeting_auto`) belong to this week; whether the `*_score` tables
   really differ from the `*_auto` ones.
4. **Writes the date folder** `data/weekly/<D>/` (5 sub-folders, `.csv.gz`, the target lists, a fresh
   `_manifest.json` with D and T). Your `data_raw/` is never changed.
5. **Cleans the duplicates with the NBA rule** (step 4b; notebook 01 explains where they come from).
6. **`TEST_10K = True`**: also writes `data/weekly_sample/<D>/`: **10,000 training lines** + **10,000 lines of ONE
   target group** (`SCORE_GROUP`), for notebook 03 / 04 with `TEST_10K = True`.

Run with the working environment (`.venv` activated, `python -m jupyterlab` from the project folder): the contract
and the calendar are read from the project code (`nbx/fsdr`), the same code as Part A of notebooks 03 / 04.

In [ ]:
# --- settings -------------------------------------------------------------------
RAW_DIR = "data/data_raw"         # the flat folder where you paste every exported table
RUN_DATE = None                   # D = t1 (scoring); None = the most recent day of the data (the last purchase of subs_btl);
                                  #   or a date "2026-11-19". 03 / 04 then take the latest week prepared here
OVERWRITE = True                  # data/weekly/<D>/ exists: False = stop; True = move it to data/backup/ and rebuild
FIX_HEADERS = True                # a file without header: add the contract header when the column count and the IDs match
NBA_DEDUP = True                  # one row per key in data/weekly/<D>/, with the NBA recipe's rule (step 4b)
NBA_STRICT = False                # True = also drop the rows of another account / subscription than base_prepaid's
REV_DECREASE_OPTIONAL = False     # True only if REV_DECREASE_204_SCORING_TD.predictions is not exported (03 / 04 then run without it)

# target lists: target2_a / target2_b files in data_raw, or the NBA sample table with a "group" column
NBA_SAMPLE_TABLE = "new_tg_nba_28oct26"            # the NBA recipe's target sample (used if no target2_a / target2_b)
GROUP_MAP = {"cstr": "t2c", "free": "t2f", "free_w_nat": "t2f"}   # its group -> T2C (a, constraint) / T2F (b, free)

# --- test on 10k ----------------------------------------------------------------
RUN_SIZE = "all"                  # "all" = the full week only (~50k per list): for 04_..._real25k (whole base to
                                  #   train, its own RUN_SIZE scored) and for 03 / 04 with RUN_SIZE = "all"
                                  #   | 1_000 | 5_000 | 10_000 = also a test copy for 03 / 04 (the SAME RUN_SIZE there)
TEST_10K = RUN_SIZE != "all"      # (from RUN_SIZE)
LAST_CAMPAIGN_DAYS = 7            # targeting_dataiku_auto holds the history: keep the latest campaign only (its last 7 days)
# --- the dates (notebooks3): t1 = D = the most recent date of the data, t2 = T = t1 - 7 ------------------------------
ANCHOR_LAG_DAYS = 7               # T = D - 7: the *_auto tables (training) are one week before the *_score tables (scoring)
REWARD_WINDOW = (-7, 0)           # last week's purchases are counted from t2 = D - 7 to t1 = D (engine default: D-7 .. D-1)
TARGETING_WINDOW = (-7, -1)       # last week's campaign (date1) is expected from t2 to t1 - 1 (engine default: D-8 .. D-2)
TRAIN_LINES = 10_000              # test copy: lines of your real base_prepaid used to train (agent 2b learns from them)
SCORE_LINES = None if RUN_SIZE == "all" else int(RUN_SIZE)   # test copy: the target lines scored (from RUN_SIZE)
SCORE_GROUP = "t2c"               # "t2c" = target2/a_constraint/target2_a  |  "t2f" = target2/b_free/target2_b
TEST_REPAIRS = True               # repair the COPY only (unused contract columns, empty keys), each repair listed
SEED = 7
CHUNK = 500_000

In [ ]:
# --- the step timer: the time of every step, its estimate and the time left (do not edit) ------------------------------
import json as _json, time as _time2
from datetime import datetime as _dt, timedelta as _td
from pathlib import Path as _Path

_root = _Path.cwd()
while not (_root / "pyproject.toml").exists() and _root != _root.parent:
    _root = _root.parent
STEP_FILE = _root / "data" / "runs" / "_step_times.json"        # the times of the earlier runs (step names only, no MSISDN)
STEP_ORDER = ['Step 1', 'Step 2', 'Step 3', 'Step 4', 'Step 4b', 'Step 4c', 'Step 5', 'Time per step']
STEP_DEFAULT = {'Step 1': (2, 2), 'Step 2': (0.5, 0.5), 'Step 3': (5, 5), 'Step 4': (15, 15), 'Step 4b': (10, 10), 'Step 4c': (1, 1), 'Step 5': (0.1, 5)}    # minutes (full week, test copy): only for the first run, then your own measured times
_NB, _SIZE = "02_prepare_week", str(RUN_SIZE) + ""
try:
    _HIST = _json.loads(STEP_FILE.read_text(encoding="utf-8")).get(_NB, {}).get(_SIZE, {})
except (OSError, ValueError):
    _HIST = {}
STEP = {"cur": None, "t0": None, "done": {}, "start": _time2.time()}


def _mm(s):
    s = int(round(s or 0))
    return f"{s // 3600}h{s % 3600 // 60:02d}m" if s >= 3600 else f"{s // 60}m{s % 60:02d}s"


def _estimate(key):
    """Seconds: agents 3 / 4 from step 5b once it ran, else the time of your last run of the same size, else the default."""
    g = globals()
    if key in ("Step 6", "Step 7") and "U" in g and "keep" in g and "s_nba" in g:
        kept = g["U"][g["keep"]]
        nba = float((kept["group"] == "t2c").sum()) * g["s_nba"]
        if key == "Step 6":
            return nba
        cpu = _HIST.get("Step 7 (CPU)", 60.0 * STEP_DEFAULT["Step 7 (CPU)"][0 if RUN_SIZE == "all" else 1])
        return float(kept["sec"].sum()) - nba + cpu
    if key in _HIST:
        return float(_HIST[key])
    d = STEP_DEFAULT.get(key)
    return 60.0 * (d[0] if RUN_SIZE == "all" else d[1]) if d else 0.0


def _save():
    try:
        allh = _json.loads(STEP_FILE.read_text(encoding="utf-8")) if STEP_FILE.exists() else {}
        allh.setdefault(_NB, {})[_SIZE] = {**_HIST, **{k: round(v, 1) for k, v in STEP["done"].items()}}
        STEP_FILE.parent.mkdir(parents=True, exist_ok=True)
        STEP_FILE.write_text(_json.dumps(allh, indent=1), encoding="utf-8")
    except (OSError, ValueError):
        pass


def _close():
    if STEP["cur"]:
        dur = _time2.time() - STEP["t0"]
        STEP["done"][STEP["cur"]] = dur
        print(f"[time] {STEP['cur']} done in {_mm(dur)} (estimate {_mm(_estimate(STEP['cur']))})")
        _save()
    STEP["cur"] = None


def step(key):
    """At the top of each step: closes the previous step, prints this step's estimate and the time left."""
    _close()
    STEP["cur"], STEP["t0"] = key, _time2.time()
    rest = STEP_ORDER[STEP_ORDER.index(key):] if key in STEP_ORDER else [key]
    left = sum(_estimate(k) for k in rest)
    print(f"[time] {key}: estimate {_mm(_estimate(key))} | elapsed {_mm(_time2.time() - STEP['start'])} | "
          f"left about {_mm(left)} (end about {_dt.now() + _td(seconds=left):%H:%M})")


def step_report():
    """The time of every step against its estimate (the last cell)."""
    _close()
    rows = [{"step": k, "minutes": round(STEP["done"][k] / 60, 1) if k in STEP["done"] else None,
             "estimate (min)": round(_estimate(k) / 60, 1)} for k in STEP_ORDER]
    import pandas as _pd
    t = _pd.DataFrame(rows)
    total = (_time2.time() - STEP["start"]) / 60
    print(f"total {total:.1f} min (RUN_SIZE {RUN_SIZE}); the next run of the same size uses these times as its estimates "
          f"({STEP_FILE.name})")
    return t


print(f"step timer ready: {len(STEP_ORDER)} steps, estimates from "
      + ("your last run of this size" if _HIST else "the defaults (first run of this size)"))


In [ ]:
import csv, difflib, gzip, io, json, os, re, shutil, sys, time
from datetime import date, datetime, timedelta
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import Markdown, display

ROOT = Path.cwd()
while not (ROOT / "pyproject.toml").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
os.chdir(ROOT)
sys.path.insert(0, str(ROOT))
import nbx
from nbx.fsdr import config, schemas
from nbx.fsdr import calendar as cal

pd.set_option("display.max_columns", 30, "display.width", 220, "display.max_colwidth", 110, "display.max_rows", 200)
RAW = ROOT / RAW_DIR
assert RAW.is_dir(), f"no folder {RAW}: create it and paste the exported tables"
LIST_KEYS = {"t2c": "target2/a_constraint/target2_a", "t2f": "target2/b_free/target2_b"}
SIZE_ENV = {"t2c": "FSDR_T2_CONSTRAINT_SIZE", "t2f": "FSDR_T2_FREE_SIZE"}
assert SCORE_GROUP in LIST_KEYS, "SCORE_GROUP must be 't2c' or 't2f'"
print("engine code:", nbx.__file__, "\nraw folder :", RAW)

In [ ]:
# --- what the flux really reads (from the engine code, nbx/fsdr) ---------------------------------------------------
# Column names in UPPER case (most readers upper-case the columns first). A regex entry covers the period columns.
# Tables with an empty set are loaded and checked by T0, but no step of the flux reads their columns.
import re

_P = r"(W[1-4]|M[1-3])"
USED = {
    "base_prepaid": {"ACCESS_METHOD_VAL", "ACCESS_METHOD_ID", "ACCOUNT_ID", "PARTY_ID", "ACCESS_METHOD_SUBS_START_DT",
                     "CURRENT_PLAN"},
    "consumer_aact_account": {"ACCOUNT_ID", "PARTY_ID"},
    "access_method_id_auto": set(),                       # only the T0 cross-checks read it
    "party_auto": {"PARTY_ID", "NATIONALITY", "AGE"},
    "party_scoring": {"PARTY_ID", "NATIONALITY", "AGE"},
    "customer_profile": {"MSISDN", "AGE", "TENURE"},
    "location_auto": {"ACCESS_METHOD_ID", "REGION", "CALLING_MONTH", "CALLING_YEAR"},
    "DEVICEBEHAVIORSEGMENTATION.dv_segment": {"ACCESS_METHOD_ID", "DV_SEGMENT"},
    "paygo_activity_auto": set(),                         # "loaded but not used"
    "subs_amt_auto": {"ACCESS_METHOD_VAL", "ACCESS_METHOD_ID", rf"SUB_AMT_{_P}"},
    "subs_amt_score": {"ACCESS_METHOD_VAL", "ACCESS_METHOD_ID", rf"SUB_AMT_{_P}"},
    "subscriptions_tg": {"ACCESS_METHOD_VAL", "ACCESS_METHOD_ID", rf"(TOTAL|DATA|VOICE)_SUB_AMT_{_P}"},
    "usage_data_voice_payg_tg": {"ACCESS_METHOD_VAL", "ACCESS_METHOD_ID", rf"PG_TOTAL_AMOUNT_{_P}"},
    "targeting_dataiku_auto": {"ACCESS_METHOD_ID", "DATE1", "SEGMENT"},
    "targeting_auto": {"MSISDN", "START_DATE"},
    "subs_btl_auto": {"ACCESS_METHOD_VAL", "PRODUCTNAME", "FULFILMENTSUCCESSDATE_TIME"},
    "sub_btl_3months": {"ACCESS_METHOD_VAL", "PRODUCTNAME", "NB_SUB"},
    "btl_catalog": {"BTL_NAME", "TYPE", "SERVICE", "GENEROSITY", "VALIDITY", "PRICE", "GB", "MIN", "PRICE DATA",
                    "PRICE VOICE"},
    "BTL_script_english_arabic_ok": {"DESTINATION_OFFER_NAME", "ENGLISH SCRIPT", "ARABIC SCRIPT"},
    "conflicting_offers_h": {"OFFER_A", "OFFER_B"},
    "consent_contact": {"ACCESS_METHOD_VAL", "DND", "MARKETING_OPT_IN"},
    "sample_cg_tg_new_7oct2026": {"ACCESS_METHOD_VAL", "GROUP"},
}
for _p in ("w1", "w2", "w3", "w4", "m1", "m2", "m3"):
    _cols = {"ACCESS_METHOD_VAL", "ACCESS_METHOD_ID",
             rf"(LOCAL_VOICE_MIN|DATA_VOLUME_MB|PG_DATA_AMOUNT|PG_TOTAL_AMOUNT|ACTIVE_DAYS_COUNT)_{_P}"}
    USED["activities_auto_m2" if _p == "m2" else f"activities_{_p}_auto"] = _cols
    USED[f"activities_{_p}_score"] = _cols
MODEL_USED = {"ACCESS_METHOD_VAL", "ACCESS_METHOD_ID", "PROBA_1"}      # the 11 model-score files
# readers that use the exact column name (no upper-case): a case-only difference breaks them
EXACT_CASE = {"sub_btl_3months", "conflicting_offers_h", "targeting_auto"}
# what each table is used for (why a problem there matters)
PURPOSE = {
    "base_prepaid": "the base: every line of the run", "consumer_aact_account": "party of a line when PARTY_ID is empty",
    "party_auto": "nationality and age (training, T)", "party_scoring": "nationality and age (scoring view, D)",
    "customer_profile": "age and tenure bands",
    "subs_amt_auto": "revenue features (training, T)", "subs_amt_score": "revenue features (scoring view, D)",
    "subscriptions_tg": "price anchors of the offers",
    "usage_data_voice_payg_tg": "price anchors of the offers (PAYG part) + F-05", "targeting_dataiku_auto": "last campaign: segment and action",
    "targeting_auto": "H-05: lines already in another campaign", "subs_btl_auto": "last campaign rewards (who took an offer)",
    "sub_btl_3months": "offer universe + favourites + BTL taker flag", "btl_catalog": "the offers",
    "conflicting_offers_h": "pairs of offers never sent together", "BTL_script_english_arabic_ok": "SMS scripts of the final file",
    "consent_contact": "do-not-contact filter", "location_auto": "REGION (context only)",
    "DEVICEBEHAVIORSEGMENTATION.dv_segment": "DV_SEGMENT (context only)",
}


def used_columns(table, model_tables=()):
    return MODEL_USED if table in model_tables else USED.get(table)


def is_used(table, column, model_tables=()):
    """True / False, or None when the table is not in the map."""
    cols = used_columns(table, model_tables)
    if cols is None:
        return None
    c = str(column).upper()
    return any(re.fullmatch(u, c) for u in cols)


NUM = re.compile(r"^-?\d+([.,]\d+)?$|^\d{4}-\d{2}-\d{2}")
SUFFIX = re.compile(r"_(hdfs|hfds)$", re.I)


def strip_ext(name: str) -> str:
    for suf in (".csv.gz", ".csv"):
        if name.lower().endswith(suf):
            return name[: -len(suf)]
    return Path(name).stem


def first_lines(p: Path, n: int = 1) -> list[list[str]]:
    opener = gzip.open if p.name.lower().endswith(".gz") else open
    with opener(p, "rt", encoding="utf-8-sig", errors="replace", newline="") as f:
        lines = [f.readline() for _ in range(n)]
    return [r for r in csv.reader(io.StringIO("".join(lines)))]


def headerless(header: list[str]) -> bool:
    vals = [h.strip() for h in header if h.strip()]
    return bool(vals) and sum(bool(NUM.match(v)) for v in vals) / len(vals) >= 0.5


def read_csv(p: Path, **kw) -> pd.DataFrame:
    return pd.read_csv(p, dtype=str, keep_default_na=False, encoding="utf-8-sig", **kw)


def canon_id(s: pd.Series) -> pd.Series:
    return s.astype(str).str.strip().str.replace(r"\.0+$", "", regex=True)


def canon_val(s: pd.Series) -> pd.Series:
    s = canon_id(s)
    return s.where(~s.str.fullmatch(r"\d{9,}"), s.str[-8:])        # country prefix -> the 8 digits


def find(cols, *names) -> list[str]:
    up = {str(c).upper(): c for c in cols}
    return [up[n] for n in names if n in up]


CONTRACT = list(schemas.TABLES)


def contract_name(stem: str) -> tuple[str | None, str]:
    s = SUFFIX.sub("", stem)
    s = schemas.TABLE_ALIASES.get(s, s)
    if s in schemas.TABLES:
        return s, "" if s == stem else f"suffix removed ({stem})"
    ci = [c for c in CONTRACT if c.lower() == s.lower()]
    if ci:
        return ci[0], f"case fixed ({stem})"
    close = difflib.get_close_matches(s, CONTRACT, n=1, cutoff=0.85)
    return (close[0], f"typo fixed ({stem}) -- CHECK") if close else (None, "")

In [ ]:
# --- notebooks3: the new export names and the dates t1 / t2 (applied here, the engine is not modified) -----------------
NEW_NAMES = {'subs_btl_test': 'subs_btl_auto', 'dv_segment': 'DEVICEBEHAVIORSEGMENTATION.dv_segment', 'SEGMENT_Conflicting_Offers_h': 'conflicting_offers_h', 'REV_DECREASE_204_SCORING_predictions': 'REV_DECREASE_204_SCORING_TD.predictions', 'PROPENSITY_REACTIVATION_15D_401_Scored': 'PROPENSITY_REACTIVATION_15D_401_SCORING.Scored', 'DAIM_PROPENSITY_REACTIVATION_30D_404_SCORING_Scored': 'DAIM_PROPENSITY_REACTIVATION_30D_404_SCORING_TD.Scored', 'DAIM_FRAGILITY_VOICE_SCORING_prediction': 'DAIM_FRAGILITY_VOICE_SCORING_TD.prediction', 'DAIM_FRAGILITY_DATA_SCORING_prediction': 'DAIM_FRAGILITY_DATA_SCORING_TD.prediction', 'DAIM_EARLY_CHURN_30_60_SCORING_scoring_data_scored': 'DAIM_EARLY_CHURN_30_60_SCORING_TD.scoring_data_scored', 'DAIM_EARLY_CHURN_0_30_SCORING_Early_Churn_0_30_Prediction': 'DAIM_EARLY_CHURN_0_30_SCORING_TD.Early_Churn_0_30_Prediction', 'DAIM_CHURN_GLOBAL_SCORING_prediction': 'DAIM_CHURN_GLOBAL_SCORING_TD.prediction', 'CHURN_ONE_DAY_ACTIVE_107_Churn_One_Day_Scoring': 'CHURN_ONE_DAY_ACTIVE_107_SCORING.Churn_One_Day_Scoring', 'APPETENCY_VOICE_204_SCORING_Scored': 'APPETENCY_VOICE_204_SCORING_TD.Scored', 'APPETENCY_DATA_204_SCORING_prediction': 'APPETENCY_DATA_204_SCORING_TD.prediction'}
NOT_USED = {'target_new_april26': 'the old target list (the lists are target2_a / target2_b)', 'tenure': 'tenure comes from customer_profile, else access_method_id_auto.ACTIVATION_DATE', 'DAIM_PROPENSITY_REACTIVATION_30D_404_SCORING_filterd_nationality': 'a filtered copy: the _Scored file is used'}
for _k, _v in NEW_NAMES.items():
    schemas.TABLE_ALIASES[_k] = _v
    schemas.TABLE_ALIASES[_k.lower()] = _v
for _k in NOT_USED:
    schemas.TABLE_ALIASES[_k] = schemas.TABLE_ALIASES[_k.lower()] = "__not_used__"
cal.anchor_date = lambda d: d - timedelta(days=ANCHOR_LAG_DAYS)            # T = t2 = D - 7
config.REWARD_WINDOW = tuple(REWARD_WINDOW)                                 # purchases from t2 to t1
config.TARGETING_WINDOW = tuple(TARGETING_WINDOW)                           # last week's campaign from t2 to t1 - 1
print("new export names:", len(NEW_NAMES), "| not used:", ", ".join(NOT_USED))
print(f"dates: T = D - {ANCHOR_LAG_DAYS}; last week's purchases counted D{REWARD_WINDOW[0]:+d} .. D{REWARD_WINDOW[1]:+d}")

## Step 1 · What is in `data_raw/`

In [ ]:
step("Step 1")
rows = []
for p in sorted(RAW.rglob("*")):
    if not p.is_file() or not p.name.lower().endswith((".csv", ".csv.gz")):
        if p.is_file():
            rows.append({"file": p.name, "kind": "ignored (not .csv / .csv.gz)"})
        continue
    stem = strip_ext(p.name)
    table, note = contract_name(stem)
    head = first_lines(p, 1)
    hdr = head[0] if head else []
    s0 = SUFFIX.sub("", stem).lower()
    kind = ("contract" if table else
            "target list" if re.fullmatch(r"target2_[ab]", s0) else
            "NBA target sample" if s0 == NBA_SAMPLE_TABLE.lower() else "unknown (ignored)")
    rows.append({"file": p.name, "table": table or SUFFIX.sub("", stem), "kind": kind, "note": note,
                 "MB": round(p.stat().st_size / 1e6, 1), "n_columns": len(hdr), "headerless": headerless(hdr),
                 "header": ",".join(hdr)[:110], "path": str(p)})
inv = pd.DataFrame(rows)
dups = inv[inv["kind"] == "contract"].groupby("table").filter(lambda g: len(g) > 1)
display(inv[["file", "table", "kind", "note", "MB", "n_columns", "headerless"]])
problems = []
if len(dups):
    problems.append("two files for one table: " + ", ".join(sorted(set(dups["table"]))))
    display(Markdown("**Two files for the same table** (keep one in data_raw):"), dups[["file", "table", "MB"]])
bad_hdr = inv[(inv["kind"] == "contract") & inv["headerless"]]
if len(bad_hdr):
    print("files WITHOUT header:", ", ".join(bad_hdr["file"]),
          "-> FIX_HEADERS = True adds the contract header if the sanity check passes; else re-export with a header")
print(f"\n{(inv['kind'] == 'contract').sum()} contract tables, {(inv['kind'] == 'target list').sum()} target lists, "
      f"{(inv['kind'] == 'NBA target sample').sum()} NBA sample table, {inv['kind'].str.startswith('unknown').sum()} unknown")

## Step 2 · The tables of the NBA logic: what is missing

| group | tables | used by |
|---|---|---|
| base | `base_prepaid`, `consumer_aact_account`, `access_method_id_auto` | both |
| **training** (anchor T) | `party_auto`, `subs_amt_auto`, `activities_w1…w4_auto`, `activities_m1_auto`, `activities_auto_m2`, `activities_m3_auto` | T2 / T3 |
| last campaign | `targeting_dataiku_auto`, `subs_btl_auto`, `targeting_auto` | T3, agent 2b, P1 (H-05) |
| **scoring** (anchor D) | `party_scoring`, `subs_amt_score`, `activities_w1…m3_score` | step 4b of 03 / 04, agents 3 / 4 |
| models | the 11 model files (with REV_DECREASE) | both |
| offers / final | `btl_catalog`, `BTL_script_english_arabic_ok`, `conflicting_offers_h`, `sub_btl_3months`, `subscriptions_tg`, `usage_data_voice_payg_tg` | P6, final file |

In [ ]:
step("Step 2")
SCORE_TABLES = ["party_scoring", "subs_amt_score"] + [f"activities_{p}_score" for p in ("w1", "w2", "w3", "w4", "m1", "m2", "m3")]
REQUIRED = list(dict.fromkeys(config.PIPELINE_TABLES["train_week0"]["required"] + SCORE_TABLES
                              + ["BTL_script_english_arabic_ok", "conflicting_offers_h"]))
OPTIONAL = [config.PROFILE_TABLE, config.CONSENT_TABLE, "location_auto", "DEVICEBEHAVIORSEGMENTATION.dv_segment",
            "paygo_activity_auto"]
REV_DECR_TABLE = "REV_DECREASE_204_SCORING_TD.predictions"
if REV_DECREASE_OPTIONAL and REV_DECR_TABLE in REQUIRED:
    REQUIRED.remove(REV_DECR_TABLE)
    OPTIONAL.append(REV_DECR_TABLE)
have = set(inv.loc[inv["kind"] == "contract", "table"])
req = pd.DataFrame([{"table": t, "group": ("scoring (D)" if t in SCORE_TABLES else
                                          "model" if t in schemas.model_score_tables() else
                                          "training (T)" if t.endswith("_auto") or "auto_m2" in t else "base / offers"),
                     "present": t in have} for t in REQUIRED])
print(f"required: {req['present'].sum()} / {len(req)} present")
missing_req = req.loc[~req["present"], "table"].tolist()
if missing_req:
    display(Markdown("**Missing required tables:**"), req[~req["present"]])
    problems.append(f"{len(missing_req)} required table(s) missing")
print("optional present:", [t for t in OPTIONAL if t in have], "| optional missing:", [t for t in OPTIONAL if t not in have])

## Step 3 · The two dates: D (scoring, `*_score`) and T (training, `*_auto`)

**D = t1** is read from the data: the most recent purchase day of `subs_btl_auto` (export `subs_btl_test`), unless
`RUN_DATE` is set. **T = t2 = t1 − 7** (`ANCHOR_LAG_DAYS`). `access_method_id_auto` (the newest activation) and the
latest campaign of `targeting_dataiku_auto` are shown for information. Then every window of both sets, the
share of each dated table inside its window, and a check that the `*_score` tables are not copies of the `*_auto` ones.

In [ ]:
step("Step 3")
def path_of(t):
    r = inv[(inv["kind"] == "contract") & (inv["table"] == t)]
    return Path(r.iloc[0]["path"]) if len(r) else None


def date_col(t, col):
    p = path_of(t)
    if p is None:
        return None
    hdr = first_lines(p, 1)[0]
    real = next((h for h in hdr if h.strip().lower() == col.lower()), None)
    if real is None:
        return None
    return pd.to_datetime(read_csv(p, usecols=[real])[real], errors="coerce").dt.date.dropna()


dated = {"subs_btl_auto": ("FULFILMENTSUCCESSDATE_TIME", 0),            # t1 = the most recent purchase day
         "access_method_id_auto": ("ACTIVATION_DATE", None),             # information: the newest activation
         "targeting_dataiku_auto": ("date1", None),                      # information: the history of the campaigns
         "targeting_auto": ("START_DATE", None)}
DATES, implied = {}, []
for t, (col, lag) in dated.items():
    s = date_col(t, col)
    DATES[t] = s
    if s is None or s.empty:
        implied.append({"table": t, "column": col, "rows": 0, "min": None, "max": None, "implies D": None})
        continue
    implied.append({"table": t, "column": col, "rows": len(s), "min": s.min(), "max": s.max(),
                    "implies D": s.max() + timedelta(days=lag) if lag is not None else None})
display(pd.DataFrame(implied))
cands = [r["implies D"] for r in implied if r["implies D"] is not None]
if RUN_DATE:
    D = date.fromisoformat(RUN_DATE)
    how = "set in the settings"
elif cands:
    D = cands[0]
    how = f"read from {implied[[r['implies D'] for r in implied].index(cands[0])]['table']}"
else:
    raise SystemExit("cannot read D from the data (no subs_btl_auto / targeting_dataiku_auto dates): set RUN_DATE")
T = cal.anchor_date(D)
print(f"D (run date, scoring *_score)   = {D} ({D:%A}) -- {how}")
print(f"T (anchor, training *_auto)    = {T} ({T:%A}), D - T = {(D - T).days} day(s)")
if len(set(cands)) > 1:
    print(f"WARNING: the dated tables imply different D: {sorted(set(map(str, cands)))} -> re-export them for the same week")
    problems.append("dated tables of different weeks")
today = date.today()
if (today - D).days > 10:
    print(f"WARNING: D is {(today - D).days} days before today ({today}): are these this week's exports?")

In [ ]:
def wtab(name, windows):
    return [{"table": name, "period": p, "start": w.start, "end": w.end, "days": w.days} for p, w in windows.items()]


win = pd.DataFrame(wtab("activities_*_auto (T)", cal.activity_windows(T)) + wtab("activities_*_score (D)", cal.activity_windows(D))
                   + wtab("subs_amt_auto (T)", cal.subs_windows(T)) + wtab("subs_amt_score (D)", cal.subs_windows(D)))
display(Markdown("**Windows of the training (T) and scoring (D) tables**"))
display(win.pivot_table(index="period", columns="table", values=["start", "end"], aggfunc="first")
        .swaplevel(axis=1).sort_index(axis=1).reindex(cal.PERIODS))

_s = DATES.get("targeting_dataiku_auto")
if _s is not None and len(_s):        # the table holds the history: the window check reads the latest campaign only
    _start = _s.max() - timedelta(days=LAST_CAMPAIGN_DAYS - 1)
    DATES["targeting_dataiku_auto"] = _s[_s >= _start]
    print(f"targeting_dataiku_auto: {len(_s):,} rows of history, {_s.min()} .. {_s.max()}; the latest campaign = "
          f"{len(DATES['targeting_dataiku_auto']):,} rows from {_start} (step 4c keeps only those)")
_b = DATES.get("subs_btl_auto")
if _b is not None and len(_b):        # the purchase history: the window check reads the reward window only
    _w = cal.campaign_windows(D)["reward"]
    print(f"subs_btl_auto: {len(_b):,} purchases, {_b.min()} .. {_b.max()}; {int(((_b >= _w.start) & (_b <= _w.end)).sum()):,} "
          f"in the reward window {_w.start} .. {_w.end} (the others are history, not used for the reward)")
    DATES["subs_btl_auto"] = _b[_b >= _w.start]
cw = cal.campaign_windows(D)
chk = []
for t, key in (("targeting_dataiku_auto", "targeting"), ("subs_btl_auto", "reward"), ("targeting_auto", "arbitration")):
    s, w = DATES.get(t), cw[key]
    inside = int(((s >= w.start) & (s <= w.end)).sum()) if s is not None and len(s) else 0
    n = 0 if s is None else len(s)
    chk.append({"table": t, "window": f"{w.start} .. {w.end}", "rows": n, "inside": inside,
                "share inside": round(inside / n, 3) if n else None,
                "verdict": "OK" if n and inside / n > 0.95 else ("EMPTY" if not n else "OUTSIDE: another week?")})
    if n and inside / n <= 0.95:
        problems.append(f"{t}: {n - inside:,} rows outside {w.start}..{w.end}")
display(Markdown("**Dated tables vs the windows of D** (BR-21, BR-23, BR-22)"), pd.DataFrame(chk))

In [ ]:
# the *_score tables must be the D version, not a copy of the *_auto tables
def sample_col(t, col, n=300_000):
    p = path_of(t)
    if p is None:
        return None
    df = read_csv(p, nrows=n)
    v, c = find(df.columns, "ACCESS_METHOD_VAL"), find(df.columns, col.upper())
    if not v or not c:
        return None
    return pd.Series(pd.to_numeric(df[c[0]], errors="coerce").to_numpy(), index=canon_val(df[v[0]])).groupby(level=0).sum()


same = []
for auto, score, col in (("activities_w1_auto", "activities_w1_score", "LOCAL_VOICE_MIN_W1"),
                         ("activities_m1_auto", "activities_m1_score", "DATA_VOLUME_MB_M1"),
                         ("subs_amt_auto", "subs_amt_score", "SUB_AMT_M1")):
    a, s = sample_col(auto, col), sample_col(score, col)
    if a is None or s is None:
        same.append({"pair": f"{auto} vs {score}", "column": col, "common lines": 0, "identical": None, "verdict": "missing"})
        continue
    common = a.index.intersection(s.index)
    eq = float((a.loc[common].round(3) == s.loc[common].round(3)).mean()) if len(common) else None
    verdict = ("same days (D = T): identical expected" if D == T else
               "IDENTICAL although D != T: the *_score export looks like a copy of *_auto" if eq is not None and eq > 0.99 else "OK (differ)")
    if "IDENTICAL" in verdict:
        problems.append(f"{score} looks like a copy of {auto}")
    same.append({"pair": f"{auto} vs {score}", "column": col, "common lines": len(common),
                 "identical": None if eq is None else round(eq, 3), "verdict": verdict})
display(Markdown("**`*_auto` vs `*_score`** (first 300k rows of each)"), pd.DataFrame(same))

## Step 4 · Write the date folder `data/weekly/<D>/`

Each table is copied to its sub-folder under its contract name as `.csv.gz` (a `.csv` is compressed), the target
lists in the same folders as always, `target2/a_constraint/target2_a.csv.gz` and `target2/b_free/target2_b.csv.gz`
(taken from `data_raw`, or **your lists already in `data/weekly/<D>/target2/`, kept as they are**, or the NBA sample;
a headerless one-column list is read as `access_method_val`, 8 digits, `968` prefix dropped), and a fresh
`_manifest.json` (D, T, source = data_raw). `data_raw/` is not changed.

In [ ]:
step("Step 4")
DEST = config.FSDR_WEEKLY_DIR / D.isoformat()
if problems:
    print("PROBLEMS FOUND (the folder is still written; notebook 03 / 04 step 2 will report them again):")
    print("\n".join(f"- {p}" for p in problems))
LIST_DIRS = ("target1", "target2", "control")


def table_files(folder: Path) -> list[Path]:
    return [p for p in folder.rglob("*.csv.gz") if p.relative_to(folder).parts[0] not in LIST_DIRS]


if DEST.exists():
    if table_files(DEST):
        if not OVERWRITE:
            raise SystemExit(f"{DEST} already holds tables: set OVERWRITE = True (moved to data/backup/, your target "
                             "folders are kept) or delete it")
        bak = ROOT / "data" / "backup" / f"weekly_{D.isoformat()}_{datetime.now():%Y%m%d_%H%M%S}"
        bak.parent.mkdir(parents=True, exist_ok=True)
        shutil.move(str(DEST), str(bak))
        print("previous folder moved to", bak)
        for d in LIST_DIRS:                      # your target folders come back to the same place
            if (bak / d).is_dir():
                shutil.copytree(bak / d, DEST / d)
    elif any(DEST.iterdir()):
        print(f"{DEST} holds only your target folders: they are kept in place")
DEST.mkdir(parents=True, exist_ok=True)


def write_table(src: Path, dst: Path, header: list[str] | None = None):
    dst.parent.mkdir(parents=True, exist_ok=True)
    gz_in = src.name.lower().endswith(".gz")
    if header is None and gz_in:
        shutil.copy2(src, dst)
        return
    with (gzip.open(src, "rb") if gz_in else open(src, "rb")) as fi, gzip.open(dst, "wb", compresslevel=6) as fo:
        if header is not None:
            fo.write((",".join(header) + "\n").encode("utf-8"))
        shutil.copyfileobj(fi, fo, 16 * 1024 * 1024)


written = []
for r in inv[inv["kind"] == "contract"].drop_duplicates("table").itertuples():
    t, src = r.table, Path(r.path)
    header = None
    if r.headerless:
        names = schemas.TABLES[t].field_names
        ok = FIX_HEADERS and len(names) == r.n_columns
        if ok:
            s = pd.read_csv(src, header=None, names=names, nrows=1000, dtype=str, keep_default_na=False)
            for col, rule in (("ACCESS_METHOD_VAL", r"\d{8}"), ("ACCESS_METHOD_ID", r"\d+"), ("access_method_val", r"\d{8}")):
                if col in s.columns:
                    ok &= s[col].str.strip().str.fullmatch(rule).mean() >= 0.95
        header = names if ok else None
        print(f"{t}: no header -> {'contract header added (sanity OK)' if ok else 'copied WITHOUT header (fix it)'}")
    dst = DEST / schemas.TABLES[t].folder / f"{t}.csv.gz"
    write_table(src, dst, header)
    written.append(t)
print(f"{len(written)} tables written to {DEST}")


def read_list(p: Path):
    """A target list: with a header (access_method_val / msisdn column), or headerless (one MSISDN per line).
    Returns access_method_val (+ the other columns), or None."""
    first = first_lines(p, 1)
    if not first:
        return None
    if headerless(first[0]):
        df = read_csv(p, header=None)
        df = df.rename(columns={0: "access_method_val"}).rename(columns=lambda c: f"col_{c}" if isinstance(c, int) else c)
    else:
        df = read_csv(p)
        c = find(df.columns, "ACCESS_METHOD_VAL", "MSISDN")
        if not c:
            print(f"{p.name}: no access_method_val / msisdn column ({list(df.columns)[:5]}): not used")
            return None
        df = df.rename(columns={c[0]: "access_method_val"})
    df["access_method_val"] = canon_id(df["access_method_val"]).str.replace(r"^968(?=\d{8}$)", "", regex=True)
    df = df[df["access_method_val"].str.fullmatch(r"\d{8}")].drop_duplicates("access_method_val")
    return df


lists, list_src = {}, {}
for key, rel in LIST_KEYS.items():
    f = inv[(inv["kind"] == "target list") & (inv["table"].str.lower() == Path(rel).name.lower())]
    if len(f):
        df = read_list(Path(f.iloc[0]["path"]))
        if df is not None:
            lists[key], list_src[key] = df, f"data_raw/{f.iloc[0]['file']}"
for key, rel in LIST_KEYS.items():
    p = DEST / f"{rel}.csv.gz"
    if key not in lists and p.exists():
        df = read_list(p)
        if df is not None:
            had_header = not headerless(first_lines(p, 1)[0] if first_lines(p, 1) else [])
            lists[key], list_src[key] = df, ("kept" if had_header else "kept, header added")
if not lists:
    f = inv[inv["kind"] == "NBA target sample"]
    if len(f):
        smp = read_csv(Path(f.iloc[0]["path"]))
        smp.columns = [c.lower() for c in smp.columns]
        unknown = sorted(set(smp["group"]) - set(GROUP_MAP))
        print(f"NBA sample {f.iloc[0]['file']}: groups {smp['group'].value_counts().to_dict()}"
              + (f" -- not mapped (not scored): {unknown}" if unknown else ""))
        for key in LIST_KEYS:
            part = smp[smp["group"].map(GROUP_MAP) == key]
            if len(part):
                lists[key], list_src[key] = part, f"NBA sample {f.iloc[0]['file']}"
for key, df in lists.items():
    dst = DEST / f"{LIST_KEYS[key]}.csv.gz"
    if list_src[key] != "kept":                          # "kept, header added" is rewritten with its header
        dst.parent.mkdir(parents=True, exist_ok=True)
        df.to_csv(dst, index=False, compression={"method": "gzip", "compresslevel": 6})
    print(f"target list {key}: {len(df):,} lines, {dst.relative_to(ROOT)} "
          f"({'your file, kept as it is' if list_src[key] == 'kept' else 'from ' + list_src[key]})")
if not lists:
    print("no target list found (data_raw, data/weekly/<D>/target2/ or the NBA sample): P1 will draw the panel at random")

man = {"run_date": D.isoformat(), "anchor_date": T.isoformat(), "source": "data_raw", "raw_dir": str(RAW),
       "created": datetime.now().isoformat(timespec="seconds"), "tables": sorted(written),
       "lists": {k: int(len(v)) for k, v in lists.items() if v is not None}, "problems": problems,
       "windows": cal.all_windows(D)}
(DEST / config.MANIFEST_FILE).write_text(json.dumps(man, indent=2, default=str), encoding="utf-8")
print(f"\n_manifest.json: run_date {D}, anchor_date {T} -> notebook 03 / 04 with RUN_DATE = \"{D.isoformat()}\"")

## Step 4b · Duplicates, removed with the NBA recipe's rule (`NBA_DEDUP = True`)

The NBA recipe LEFT JOINs each table on the line and keeps the **first row per line** (`drop_duplicates`). The FSDR
engine instead **sums** the rows of a line (activities, subs_amt, price anchors), so a duplicated row doubles the line's
revenue. This step applies the recipe's rule to every table of `data/weekly/<D>/`, so the engine sees what the recipe
sees:

| tables | one row per | which row is kept |
|---|---|---|
| `activities_*`, `subs_amt_*`, `paygo_activity_auto`, `subscriptions_tg`, `usage_data_voice_payg_tg`, the model files | line (ACCESS_METHOD_VAL) | the row with the line's ACCESS_METHOD_ID **and** ACCOUNT_ID in base_prepaid, else its ACCESS_METHOD_ID, else the first |
| `targeting_dataiku_auto` | (access_method_id, date1) | a row with a real segment (not Nursering) first, else the first |
| `consumer_aact_account` / `party_*` / `DEVICEBEHAVIORSEGMENTATION` / `base_prepaid` | ACCOUNT_ID / PARTY_ID / ACCESS_METHOD_ID / line | the first (accounts of one party are all kept) |
| the other tables | their contract key | the first |

Keys are compared as the engine reads them (`0123` = `123`, `123.0` = `123`, `968` + 8 digits = the 8 digits); rows with
an empty key are left to the engine. `NBA_STRICT = True` also drops the rows that do not belong to the line in
base_prepaid (activities: other account; the others: other ACCESS_METHOD_ID), exactly as the recipe's joins. Only files
with duplicates are rewritten; `data_raw/` is never changed. **Notebook 01** explains where the duplicates come from.

In [ ]:
step("Step 4b")
# --- duplicate rules of the NBA recipe (shared by notebooks 01 and 02) ---------------------------------------------
# The NBA recipe never de-duplicates a source table: it LEFT JOINs every table on the line, then keeps the FIRST row
# per line (drop_duplicates(["ACCESS_METHOD_VAL"]) in training, ["access_method_val", "access_method_id"] in scoring).
# Join keys of the recipe:
#   activities_*            (ACCESS_METHOD_VAL, ACCESS_METHOD_ID, ACCOUNT_ID)  -> only the row of the line's account
#   subs_amt_*, the models  (ACCESS_METHOD_VAL, ACCESS_METHOD_ID)              -> the row of the line's subscription
#   party_auto / _scoring   PARTY_ID          consumer_aact_account  ACCOUNT_ID
#   DEVICEBEHAVIOR / location   ACCESS_METHOD_ID          targeting_dataiku_auto  access_method_id (-> msisdn)
# So, per table, ONE row per key, the row matching the line in base_prepaid first, else the first in file order.
# The FSDR engine SUMS the rows of a line (activities, subs_amt, price anchors): applying the NBA rule before the
# engine gives the same numbers as the recipe and never double-counts.
import numpy as np
import pandas as pd

LINE_KEY = ("ACCESS_METHOD_VAL", "ACCESS_METHOD_ID", "ACCOUNT_ID")


def canon_key(s: pd.Series, col: str) -> pd.Series:
    """A key column as the engine compares it: trimmed, '123.0' -> '123', '00123' -> '123', MSISDN = its 8 digits."""
    x = s.astype(str).str.strip().str.replace(r"\.0+$", "", regex=True)
    x = x.where(~x.str.lower().isin(["nan", "none", "null"]), "")
    if col.upper() in ("ACCESS_METHOD_VAL", "MSISDN"):
        x = x.where(~x.str.fullmatch(r"\d{9,}"), x.str[-8:])
    digits = x.str.fullmatch(r"\d+")
    return x.where(~digits, x.str.lstrip("0").replace("", "0"))


def is_key_col(c) -> bool:
    return str(c).upper().endswith(("_VAL", "_ID", "MSISDN")) or str(c).upper() == "DATE1"


def canonical_frame(df: pd.DataFrame) -> pd.DataFrame:
    """Every column as compared for duplicates: keys canonical (canon_key), the other columns trimmed text."""
    return pd.DataFrame({c: (canon_key(df[c], c) if is_key_col(c) else df[c].astype(str).str.strip()) for c in df.columns},
                        index=df.index)


def col_of(columns, name):
    return next((c for c in columns if str(c).upper() == name.upper()), None)


def table_rule(t: str, columns, schemas) -> tuple[str, list[str]]:
    """('line', [VAL, ID, ACC present]) | ('targeting', [id, date1]) | ('key', [cols]) | ('none', [])."""
    up = {str(c).upper() for c in columns}
    pk = [c for c in (schemas.TABLES[t].primary_key if t in schemas.TABLES else ()) if c.upper() in up]
    models = set(schemas.model_score_tables())
    if t in models or {k.upper() for k in pk} == set(LINE_KEY):
        return "line", [col_of(columns, k) for k in LINE_KEY if col_of(columns, k)]
    if t == "targeting_dataiku_auto":
        return "targeting", [c for c in (col_of(columns, "ACCESS_METHOD_ID"), col_of(columns, "DATE1")) if c]
    nba_key = {"base_prepaid": ["ACCESS_METHOD_VAL"], "consumer_aact_account": ["ACCOUNT_ID"],
               "party_auto": ["PARTY_ID"], "party_scoring": ["PARTY_ID"],
               "DEVICEBEHAVIORSEGMENTATION.dv_segment": ["ACCESS_METHOD_ID"], "customer_profile": ["MSISDN"]}.get(t)
    cols = [col_of(columns, k) for k in nba_key] if nba_key else [col_of(columns, k) for k in pk]
    cols = [c for c in cols if c]
    return ("key", cols) if cols else ("none", [])


def base_map(base: pd.DataFrame) -> pd.DataFrame:
    """base_prepaid -> one row per line: VAL (index), ID, ACC (canonical). First row per line, as the recipe."""
    v, i, a = (col_of(base.columns, k) for k in LINE_KEY)
    m = pd.DataFrame({"VAL": canon_key(base[v], "ACCESS_METHOD_VAL"), "ID": canon_key(base[i], "ACCESS_METHOD_ID"),
                      "ACC": canon_key(base[a], "ACCOUNT_ID") if a else ""})
    return m[m["VAL"] != ""].drop_duplicates("VAL").set_index("VAL")


def row_facts(df: pd.DataFrame, t: str, schemas, bmap: pd.DataFrame | None):
    """Per row of one chunk: group key hash, priority (lower = kept first), row hash (all columns, canonical keys),
    empty key (never removed: the engine drops those rows itself).
    line tables: key = the line; priority 0 = matches the base line's ID and ACCOUNT, 1 = its ID, 2 = no match.
    targeting: key = (access_method_id, date1); priority 0 = a segment outside Nursering, 1 = other.
    key tables: key = the NBA join key; priority 0."""
    kind, cols = table_rule(t, df.columns, schemas)
    norm = canonical_frame(df)
    row_h = pd.util.hash_pandas_object(norm, index=False).to_numpy()
    prio = np.zeros(len(df), dtype=np.int8)
    if kind == "line":
        vcol = col_of(df.columns, "ACCESS_METHOD_VAL")
        key_h = pd.util.hash_pandas_object(norm[[vcol]], index=False).to_numpy()
        if bmap is not None and len(bmap):
            b = bmap.reindex(norm[vcol])
            icol, acol = col_of(df.columns, "ACCESS_METHOD_ID"), col_of(df.columns, "ACCOUNT_ID")
            id_ok = (norm[icol].to_numpy() == b["ID"].to_numpy()) if icol else np.ones(len(df), bool)
            acc_ok = (norm[acol].to_numpy() == b["ACC"].to_numpy()) if acol else np.ones(len(df), bool)
            prio = np.where(id_ok & acc_ok, 0, np.where(id_ok, 1, 2)).astype(np.int8)
    elif kind in ("targeting", "key") and cols:
        key_h = pd.util.hash_pandas_object(norm[cols], index=False).to_numpy()
        if kind == "targeting":
            seg = col_of(df.columns, "SEGMENT")
            if seg:
                s = df[seg].astype(str).str.strip()
                bad = s.isin(["", "nan", "None"]) | s.str.startswith("Nursering")
                prio = bad.to_numpy().astype(np.int8)
    else:
        key_h = row_h
    if kind == "line":
        empty = (norm[col_of(df.columns, "ACCESS_METHOD_VAL")] == "").to_numpy()
    elif kind in ("targeting", "key") and cols:
        empty = (norm[cols] == "").any(axis=1).to_numpy()
    else:
        empty = np.zeros(len(df), dtype=bool)
    return kind, cols, key_h, prio, row_h, empty


def keep_first(key_h: np.ndarray, prio: np.ndarray, empty: np.ndarray | None = None) -> np.ndarray:
    """Bool mask: one row per key, the lowest priority first, then the first in file order (the recipe's rule).
    Rows with an empty key are all kept."""
    n = len(key_h)
    order = np.lexsort((np.arange(n), prio, key_h))          # by key, then priority, then position
    first = np.ones(n, dtype=bool)
    first[1:] = key_h[order][1:] != key_h[order][:-1]
    keep = np.zeros(n, dtype=bool)
    keep[order[first]] = True
    if empty is not None:
        keep |= empty
    return keep


In [ ]:
def nba_dedup_file(p, t, bmap):
    """two passes in chunks: 1) key / priority of each row, 2) only the kept rows written (same column order)"""
    parts, kind = [], None
    for ch in read_csv(p, chunksize=CHUNK):
        kind, cols, key_h, prio, row_h, empty = row_facts(ch, t, schemas, bmap)
        parts.append((key_h, prio, empty))
    if not parts:
        return None
    key_h = np.concatenate([x[0] for x in parts]); prio = np.concatenate([x[1] for x in parts])
    empty = np.concatenate([x[2] for x in parts])
    keep = keep_first(key_h, prio, empty)
    n_dup = int((~keep).sum())
    n_strict = 0
    if NBA_STRICT and kind == "line" and bmap is not None:
        need = 0 if t.startswith("activities") else 1           # activities join on the account too
        off = (prio > need) & keep & ~empty
        n_strict = int(off.sum())
        keep &= ~off
    if n_dup or n_strict:
        tmp = p.with_name(p.name + ".tmp")
        pos, first = 0, True
        for ch in read_csv(p, chunksize=CHUNK):
            ch[keep[pos:pos + len(ch)]].to_csv(tmp, mode="w" if first else "a", header=first, index=False,
                                               compression={"method": "gzip", "compresslevel": 6})
            pos, first = pos + len(ch), False
        os.replace(tmp, p)
    return {"table": t, "rule": kind, "rows": len(keep), "duplicates removed": n_dup,
            "not the base line (NBA_STRICT)": n_strict, "rows after": int(keep.sum()),
            "kept rows matching the base line %": round(100 * (prio[keep] == 0).mean(), 1) if kind == "line" and bmap is not None else None}


def nba_dedup_frame(df, t, bmap):
    kind, cols, key_h, prio, row_h, empty = row_facts(df, t, schemas, bmap)
    keep = keep_first(key_h, prio, empty)
    if NBA_STRICT and kind == "line" and bmap is not None:
        keep &= ~((prio > (0 if t.startswith("activities") else 1)) & ~empty)
    return pd.Series(keep, index=df.index)


dedups = []
if NBA_DEDUP:
    t0 = time.time()
    bp = next(DEST.rglob("base_prepaid.csv.gz"), None)
    BMAP = base_map(read_csv(bp, usecols=lambda c: c.upper() in LINE_KEY)) if bp else None
    for p in sorted(table_files(DEST)):
        t = strip_ext(p.name)
        if t not in schemas.TABLES or headerless(first_lines(p, 1)[0] if first_lines(p, 1) else []):
            continue
        r = nba_dedup_file(p, t, BMAP)
        if r and (r["duplicates removed"] or r["not the base line (NBA_STRICT)"]):
            dedups.append(r)
            print(f"{t}: {r['duplicates removed']:,} duplicate row(s) removed of {r['rows']:,}"
                  + (f", {r['not the base line (NBA_STRICT)']:,} not the base line" if r["not the base line (NBA_STRICT)"] else ""))
    display(Markdown(f"**Duplicates removed with the NBA rule** ({time.time() - t0:,.0f} s; why they exist: notebook 01)"),
            pd.DataFrame(dedups or [{"table": "-", "duplicates removed": 0}]))
    man["dedup"] = {"rule": "nba_first_row_per_key", "strict": NBA_STRICT, "tables": dedups}
    (DEST / config.MANIFEST_FILE).write_text(json.dumps(man, indent=2, default=str), encoding="utf-8")
    for t in ("subs_amt_auto", "subs_amt_score"):
        p = next(DEST.rglob(f"{t}.csv.gz"), None)
        if p is not None:
            v = canon_key(read_csv(p, usecols=lambda c: c.upper() == "ACCESS_METHOD_VAL").iloc[:, 0], "ACCESS_METHOD_VAL")
            v = v[v != ""]
            print(f"{t}: {len(v):,} rows for {v.nunique():,} lines = {len(v) / max(v.nunique(), 1):.2f} rows per line")
else:
    print("NBA_DEDUP = False: tables written as exported (the engine sums duplicated rows)")

## Step 4c · Last week's campaign: the latest row per line, latest campaign only

`targeting_dataiku_auto` holds **every campaign** (the history). The engine reads, per line, its most recent row as
"targeted last week" and counts its purchases from t2 to t1 as the reward. So this step keeps, per line
(`access_method_id`), **its most recent row**, and only if that row belongs to **the latest campaign** (`date1` within
`LAST_CAMPAIGN_DAYS` days of the most recent `date1`). A line targeted weeks ago is then not "targeted last week".
`data_raw/` is not changed; the counts go to `_manifest.json`.

In [ ]:
step("Step 4c")
_t = "targeting_dataiku_auto"
_p = DEST / schemas.TABLES[_t].folder / f"{_t}.csv.gz"
if not _p.exists():
    print("no targeting_dataiku_auto: no last campaign (agent 2b keeps its prior)")
else:
    _df = read_csv(_p)
    _dc, _ic = find(_df.columns, "DATE1")[0], find(_df.columns, "ACCESS_METHOD_ID")[0]
    _d1 = pd.to_datetime(_df[_dc], errors="coerce")
    _last = _d1.max()
    _start = _last - pd.to_timedelta(int(LAST_CAMPAIGN_DAYS) - 1, unit="D")
    _df = _df.assign(_D=_d1, _K=canon_id(_df[_ic])).sort_values("_D", kind="stable")
    _latest = _df.drop_duplicates("_K", keep="last")
    _keep = _latest[_latest["_D"] >= _start]
    print(f"targeting_dataiku_auto: {len(_df):,} rows, {_df['_K'].nunique():,} lines, date1 {_d1.min():%Y-%m-%d} .. {_last:%Y-%m-%d}")
    print(f"latest campaign {_start:%Y-%m-%d} .. {_last:%Y-%m-%d}: {len(_keep):,} lines kept "
          f"({len(_latest) - len(_keep):,} lines whose last targeting is older are not 'last week')")
    display(_keep["_D"].dt.date.value_counts().sort_index().rename_axis("date1").to_frame("lines kept"))
    _keep.drop(columns=["_D", "_K"]).to_csv(_p, index=False, compression={"method": "gzip", "compresslevel": 6})
    _mp = DEST / config.MANIFEST_FILE
    _m = json.loads(_mp.read_text(encoding="utf-8"))
    _m["last_campaign"] = {"from": f"{_start:%Y-%m-%d}", "to": f"{_last:%Y-%m-%d}", "rows_in_export": int(len(_df)),
                           "lines_kept": int(len(_keep)), "lines_older": int(len(_latest) - len(_keep))}
    _mp.write_text(json.dumps(_m, indent=2, default=str), encoding="utf-8")

## Step 5 · The test copy (`RUN_SIZE` = 1k / 5k / 10k): training lines + target lines of one group

Writes `data/weekly_sample/<D>/`: the `TRAIN_LINES` training lines (random eligible lines of `base_prepaid`, outside
both target lists) and `SCORE_LINES` lines of `SCORE_GROUP`'s list; every table filtered on these lines (parties and
accounts follow), the reference tables copied whole. The sample keeps the same target folders: `SCORE_GROUP`'s file
holds its scored lines, the other file is written with its header and no row, so only `SCORE_GROUP` is scored.
`_train_lines.csv` tells notebook 03 / 04 which lines agent 2b learns from.

In [ ]:
step("Step 5")
if not TEST_10K:
    print("TEST_10K = False: nothing to do (the full week is in", DEST, ")")
else:
    rng = np.random.default_rng(SEED)
    SRC, DST = DEST, ROOT / "data" / "weekly_sample" / D.isoformat()
    if DST.exists():
        shutil.rmtree(DST)
    DST.mkdir(parents=True)
    base = read_csv(next(SRC.rglob("base_prepaid.csv.gz")))
    VALc, IDc, ACCc, PARTYc = (find(base.columns, n)[0] for n in ("ACCESS_METHOD_VAL", "ACCESS_METHOD_ID", "ACCOUNT_ID", "PARTY_ID"))
    bval = canon_val(base[VALc])
    pool = pd.Index(bval[bval.str.fullmatch(r"\d{8}") & (bval > config.MSISDN_MIN_EXCLUSIVE)].unique())
    in_lists = {k: pd.Index(canon_val(v["access_method_val"]).unique()) for k, v in lists.items() if v is not None}
    if SCORE_GROUP not in in_lists:
        raise SystemExit(f"no list for {SCORE_GROUP}: TEST_10K needs the target lists")
    score_pool = in_lists[SCORE_GROUP].intersection(pool)
    score_vals = set(score_pool) if SCORE_LINES is None else set(rng.choice(score_pool, min(SCORE_LINES, len(score_pool)), replace=False))
    all_listed = pd.Index([]).append(list(in_lists.values())) if in_lists else pd.Index([])
    train_pool = pool.difference(all_listed)
    train_vals = set(rng.choice(train_pool, min(TRAIN_LINES, len(train_pool)), replace=False))
    VALS = train_vals | score_vals
    rows_b = base[bval.isin(VALS)]
    IDS, ACCOUNTS, PARTIES = (set(canon_id(rows_b[c])) - {""} for c in (IDc, ACCc, PARTYc))
    BASE_MAP = pd.DataFrame({"VAL": canon_val(rows_b[VALc]), "ID": canon_id(rows_b[IDc]), "ACC": canon_id(rows_b[ACCc])})
    print(f"training lines {len(train_vals):,} (outside the lists) + scoring lines {len(score_vals):,} of {SCORE_GROUP} "
          f"({len(score_pool):,} eligible in its list)")

    def keep_mask(df):
        ids, vals = find(df.columns, "ACCESS_METHOD_ID"), find(df.columns, "ACCESS_METHOD_VAL", "MSISDN")
        if ids or vals:
            m = pd.Series(False, index=df.index)
            for c in ids:
                m |= canon_id(df[c]).isin(IDS)
            for c in vals:
                m |= canon_val(df[c]).isin(VALS)
            return m, "+".join(ids + vals)
        acc, party = find(df.columns, "ACCOUNT_ID"), find(df.columns, "PARTY_ID")
        if acc or party:
            m = pd.Series(False, index=df.index)
            for c in acc:
                m |= canon_id(df[c]).isin(ACCOUNTS)
            for c in party:
                m |= canon_id(df[c]).isin(PARTIES)
            return m, "+".join(acc + party)
        return None, "copied whole"

    files = sorted([p for p in SRC.rglob("*.csv.gz") if p.relative_to(SRC).parts[0] not in ("target1", "target2", "control")],
                   key=lambda p: (strip_ext(p.name) != "base_prepaid", strip_ext(p.name) != "consumer_aact_account", str(p)))
    t0 = time.time()
    for p in files:
        out = DST / p.relative_to(SRC)
        out.parent.mkdir(parents=True, exist_ok=True)
        first, n_in, n_out, how = True, 0, 0, ""
        if headerless(first_lines(p, 1)[0] if first_lines(p, 1) else []):
            shutil.copy2(p, out)
            print(f"{p.name}: no header, copied whole")
            continue
        for chunk in read_csv(p, chunksize=CHUNK):
            n_in += len(chunk)
            m, how = keep_mask(chunk)
            part = chunk if m is None else chunk[m]
            n_out += len(part)
            if strip_ext(p.name) == "consumer_aact_account":
                pc = find(part.columns, "PARTY_ID")
                if pc:
                    PARTIES |= set(canon_id(part[pc[0]])) - {""}
            part.to_csv(out, mode="w" if first else "a", header=first, index=False,
                        compression={"method": "gzip", "compresslevel": 6})
            first = False
        if first:
            shutil.copy2(p, out)                                     # header-only file
        print(f"{str(p.relative_to(SRC)):72s} {n_in:>12,} -> {n_out:>8,}  [{how}]")
    # the lists: SCORE_GROUP filtered, the other group empty (not scored)
    for key, rel in LIST_KEYS.items():
        dst = DST / f"{rel}.csv.gz"
        dst.parent.mkdir(parents=True, exist_ok=True)
        df = lists.get(key)
        df = (df[canon_val(df["access_method_val"]).isin(score_vals)] if key == SCORE_GROUP
              else (df.head(0) if df is not None else pd.DataFrame(columns=["access_method_val"])))
        df.to_csv(dst, index=False, compression={"method": "gzip", "compresslevel": 6})
    pd.DataFrame({"access_method_val": sorted(train_vals)}).to_csv(DST / "_train_lines.csv", index=False)
    man_s = dict(man, source="data_raw (TEST_10K sample)", train_lines=len(train_vals), score_lines=len(score_vals),
                 score_group=SCORE_GROUP)
    (DST / config.MANIFEST_FILE).write_text(json.dumps(man_s, indent=2, default=str), encoding="utf-8")
    print(f"\n{len(files)} tables in {time.time() - t0:,.0f} s")

In [ ]:
# TEST_REPAIRS: the copy only -- contract columns the flux never reads (added empty), empty keys filled from base_prepaid
repairs = []
if TEST_10K and TEST_REPAIRS:
    from nbx.fsdr.datasource import read_table_file
    from nbx.fsdr.validation import norm_keys
    MODEL_TABLES = set(schemas.model_score_tables())
    by_val = BASE_MAP.drop_duplicates("VAL").set_index("VAL")
    by_id = BASE_MAP.drop_duplicates("ID").set_index("ID")
    SBMAP = base_map(read_csv(next(DST.rglob("base_prepaid.csv.gz")), usecols=lambda c: c.upper() in LINE_KEY))
    for p in sorted(DST.rglob("*.csv.gz")):
        t = strip_ext(p.name)
        if t not in schemas.TABLES or p.relative_to(DST).parts[0] in ("target1", "target2", "control"):
            continue
        df = read_csv(p)
        if df.empty or headerless(list(df.columns)):
            continue
        schema, changed, up = schemas.TABLES[t], False, {c.upper() for c in df.columns}
        for f in schema.fields:
            if f.name in df.columns or f.name.upper() in up:
                continue
            if is_used(t, f.name, MODEL_TABLES):
                repairs.append({"table": t, "repair": f"MISSING COLUMN READ BY THE FLUX: {f.name} (not invented)"})
                continue
            src_col = "SUBSCRIPTION_ID" if f.name == "UBSCRIPTION_ID" and "SUBSCRIPTION_ID" in df.columns else None
            df[f.name] = df[src_col] if src_col else ""
            changed = True
            repairs.append({"table": t, "repair": f"{f.name} added " + (f"(copy of {src_col})" if src_col else "empty (not read)")})
        vc, ic, ac = (find(df.columns, n) for n in ("ACCESS_METHOD_VAL", "ACCESS_METHOD_ID", "ACCOUNT_ID"))
        for col, target in ((ac, "ACC"), (ic, "ID"), (vc, "VAL")):
            if not col:
                continue
            empty = df[col[0]].str.strip() == ""
            if not empty.any():
                continue
            fill = pd.Series("", index=df.index)
            if vc and target != "VAL":
                fill = canon_val(df[vc[0]]).map(by_val[target]).fillna("")
            if ic and target != "ID":
                fill = fill.where(fill != "", canon_id(df[ic[0]]).map(by_id[target]).fillna(""))
            k = int((empty & (fill != "")).sum())
            if k:
                df.loc[empty, col[0]] = fill[empty]
                changed = True
                repairs.append({"table": t, "repair": f"{k} empty {col[0]} filled from base_prepaid"})
        keys = [f.name for f in schema.fields if f.key and f.name in df.columns]
        still = df[keys].apply(lambda s: s.str.strip() == "").any(axis=1) if keys else pd.Series(False, index=df.index)
        if still.any():
            repairs.append({"table": t, "repair": f"{int(still.sum())} row(s) with an empty key dropped"})
            df, changed = df[~still], True
        if NBA_DEDUP:                                  # a filled key can make a row a duplicate of another one
            keep = nba_dedup_frame(df, t, SBMAP)
            if (~keep).any():
                repairs.append({"table": t, "repair": f"{int((~keep).sum())} duplicate row(s) removed (NBA rule)"})
                df, changed = df[keep], True
        if changed:
            df.to_csv(p, index=False, compression={"method": "gzip", "compresslevel": 6})
        if schema.primary_key and schema.duplicates_blocking and len(df):
            # last check with the engine's own reader and key rule (what T0 of notebook 03 / 04 counts)
            raw = read_table_file(p, t)
            pk = [c for c in schema.primary_key if c in raw.columns]
            if pk and len(raw) == len(df):
                dup = raw[pk].apply(norm_keys).duplicated().to_numpy()
                if dup.any():
                    df = df[~dup]
                    df.to_csv(p, index=False, compression={"method": "gzip", "compresslevel": 6})
                    repairs.append({"table": t, "repair": f"{int(dup.sum())} duplicate key(s) removed (engine key rule)"})
    display(Markdown("**Repairs of the TEST copy** (each one = something to fix in the real exports or the contract)"),
            pd.DataFrame(repairs or [{"table": "-", "repair": "none"}]))

In [ ]:
if TEST_10K:
    from nbx.fsdr.contract import completeness
    from nbx.fsdr.datasource import JsonSource
    from nbx.fsdr.validation import validate

    spec = config.PIPELINE_TABLES["train_week0"]
    if "targeting_auto" in spec["required"]:                 # as notebook 03 / 04 in TEST_10K (this process only)
        spec["required"].remove("targeting_auto"); spec["optional"].append("targeting_auto")
    src = JsonSource(DST)
    vrep, comp = validate(src, D, "train_week0"), completeness.check(src, "train_week0", D)
    kept = {k: len(read_csv(DST / f"{rel}.csv.gz")) for k, rel in LIST_KEYS.items()}
    info = {"run_date": D.isoformat(), "anchor_date": T.isoformat(), "train_lines": len(train_vals),
            "score_lines": kept[SCORE_GROUP], "score_group": SCORE_GROUP, "lines": len(VALS),
            "env_sizes": {SIZE_ENV[k]: v for k, v in kept.items()}, "repairs": repairs, "seed": SEED,
            "missing_score_tables": [t for t in SCORE_TABLES if not src.has(t)]}
    (DST / "_sample_info.json").write_text(json.dumps(info, indent=2, default=str), encoding="utf-8")
    bl = [f"{i.table} {i.column or ''}: {i.message}" for i in vrep.issues if i.level == "blocking"] + comp["reasons"]
    print(f"sample: validation {vrep.status} {vrep.counts()}, completeness {'complete' if comp['complete'] else 'NOT complete'}")
    # handled by the settings of notebook 03 / 04 (REV_DECREASE_OPTIONAL / TARGETING_AUTO_OPTIONAL / RELAX_UNUSED_CONTRACT)
    handled = {"REV_DECREASE_204_SCORING_TD.predictions": "REV_DECREASE_OPTIONAL", "targeting_auto": "TARGETING_AUTO_OPTIONAL"}
    soft = [b for b in bl if any(t in b for t in handled)]
    bl = [b for b in bl if b not in soft]
    for b in soft:
        print(f"handled by notebook 03 / 04 ({', '.join(f for t, f in handled.items() if t in b)} = True): {b}")
    print("\n".join(["BLOCKING (T0 of notebook 03 / 04 will stop):"] + bl) if bl else
          f"nothing blocks: open notebook 03 (Ollama) or 04 (vLLM), RUN_DATE = \"{D.isoformat()}\", TEST_10K = True, Restart & Run All")
    print("lists in the sample:", kept, "-> only", SCORE_GROUP, "is scored")
else:
    print(f"full week ready in {DEST}: notebook 03 / 04 with RUN_DATE = \"{D.isoformat()}\" and TEST_10K = False")

## Time per step

The time of every step of this run against its estimate. The next run of the same `RUN_SIZE` uses these times as its estimates (`data/runs/_step_times.json`, step names and seconds only).

In [ ]:
step("Time per step")
step_report()